# A1.7 · Identity spoofing and impersonation

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.6 · Privilege compromise](https://spbreed.github.io/cyber-commons/lessons/A1.6.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Have two agents share a credential, then try to work out which one made the call.

**Why a security engineer needs it.** Attribution fails before the incident starts: you cannot say which agent acted, so you cannot revoke one without breaking all of them. The control it builds is: per-workload identity with attestation (A2.1, A2.2) and a lifecycle that can revoke one (A2.5).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Four agents share one service account. The audit log answers "what happened" perfectly and cannot answer "which one" at all — and neither can the downstream service that was deciding whether to trust the caller.

> **At CyberTravels.** All four agents share one service account, `cybertravels-svc`. The payments API can see that CyberTravels called it and cannot see which of the four — so the refund and the itinerary lookup are indistinguishable to the thing deciding whether to trust the caller. R11.

## 2 · The framework

```
   agent A --+
   agent B --+---> one service account ---> downstream service
   agent C --+          "svc-automation"          |
   agent D --+                                    v
                                        "who called me?"  -> unanswerable

   the audit log is complete and useless: every row has the same subject
```

**OWASP T9 — Identity Spoofing & Impersonation.**

A1.6 was about an agent holding too much authority. This one is about the
**identity** component being unable to say *which agent* is calling at all.

When several agents share one credential — the same service account, the same
API key baked into the same image — they are, to every downstream system, the
same principal. There is no spoofing step required. Impersonation is the
default state, because there was never a distinction to defeat.

Three consequences follow, and the third is the one that hurts during an
incident:

**Authorization cannot differ.** Every agent gets the union of what any of them
needs, which is A1.6 again, arriving through a different door.

**Attribution is impossible.** "Which agent called this?" has no answer. Not a
hard answer — no answer, because the information was never present.

**Revocation is all-or-nothing.** You have one misbehaving agent and one
credential shared by forty. Rotating it stops the incident and stops the other
thirty-nine, so the decision becomes a business call in the middle of a
response, at whatever hour it is.

In a multi-agent topology this compounds. A peer's message is trusted because it
came from a peer — but if identity cannot distinguish peers, "it came from a
peer" is a claim anyone inside the perimeter can make.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

Three agents, one credential, one incident.

## 4 · The check, as a skill

Three CyberTravels agents share one API key, so the payments API records one caller on every line. The skill measures both costs: what the record can attribute, and what revoking the key would stop.

In [ ]:
# skills/threats/shared-credential-attribution-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: shared-credential-attribution-check
description: >-
  Find credentials held by more than one agent and show what sharing does to
  attribution and to containment — who the downstream records, and what
  revoking it stops. Use when several agents call the same API, or when asked
  which agent did something and the record cannot say.
allowed-tools: Read, Grep, Glob
---

# One credential, three agents, one line in the log

A shared credential is usually adopted for convenience and paid for during an
incident. It costs two things at once: the downstream cannot attribute an
action to an agent, and the only containment available stops **every** holder.

## When to use this

Whenever more than one agent, job or replica authenticates downstream. Also
before an incident: this is the check whose absence turns a contained problem
into an outage.

## Procedure

**1 — Enumerate holders per credential.** Group by the secret, not by the
service. Environment variables, mounted files, a shared secrets-manager path
and a baked-in image layer are all the same credential when the value matches.

**2 — Read a downstream record.** Whatever the caller is identified by — an API
key id, a client id, a service account — record what the downstream can print.
If three agents map to one identifier, attribution ends there.

**3 — Simulate the destructive call.** Have one holder perform something
irreversible. Ask, from the downstream record alone, which holder did it. Write
down the answer even when it is "cannot be determined"; that sentence is the
finding.

**4 — Cost the containment.** Revoke the credential on paper and list what
stops. The count of unrelated things that stop is the number to report.

**5 — Propose per-workload identity,** and say what it costs: one credential
per agent, issued by the platform rather than pasted, so revocation is
per-agent and attribution is free.

## Output contract

```json
{
  "credentials": [{"id": "str", "holders": ["str"], "source": "env|file|manager|image"}],
  "downstream_identifier": {"field": "str", "distinguishes_holders": false},
  "destructive_probe": {"actor": "str", "recoverable_from_record": false},
  "containment": {"revoking_stops": ["str"], "collateral": 0}
}
```

## Failure modes

- **Grouping by service instead of by secret value.** Two names, one key, is
  still one credential.
- **Assuming the log will disambiguate.** Read an actual row before assuming a
  field exists.
- **Reporting only attribution.** The containment cost is what makes it urgent.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/threats/shared-credential-attribution-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/threats/shared-credential-attribution-check/scripts/shared_credential_attribution_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Show what a shared credential does to attribution and to containment when one holder misbehaves.

This is the executable half of the `shared-credential-attribution-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

SHARED_KEY = "svc-agent-7f3a1c"

AGENTS = {"triage-agent":  {"key": SHARED_KEY},
          "patch-agent":   {"key": SHARED_KEY},
          "deploy-agent":  {"key": SHARED_KEY}}

CALLS = []

def downstream(api_key, action, resource):
    """A downstream service sees only the credential presented."""
    CALLS.append({"presented": api_key, "action": action, "resource": resource})
    return {"ok": True, "caller": api_key}

for name in sorted(AGENTS):
    downstream(AGENTS[name]["key"], "read", "reports")
downstream(SHARED_KEY, "delete", "prod.customers")     # one of them did this

print("what the downstream service recorded:")
for c in CALLS:
    print(f"   caller={c['presented']}  {c['action']:7s} {c['resource']}")

incident = [c for c in CALLS if c["action"] == "delete"]
candidates = sorted(AGENTS)
print(f"\nincident: {incident[0]['action']} on {incident[0]['resource']}")
print(f"which agent did it? candidates: {candidates}")
print(f"distinguishable from the record? {len({c['presented'] for c in CALLS}) > 1}")

print("\ncontainment options:")
print(f"   rotate {SHARED_KEY} -> stops the incident, and stops "
      f"{len(AGENTS)} agents including {len(AGENTS)-1} innocent ones")
print("   rotate only the culprit -> not available; there is no 'only'")
print()
print("No attacker forged anything. Impersonation is the resting state of a")
print("system where identity was never per-workload.")
assert len({c["presented"] for c in CALLS}) == 1

## What you just proved

Three agents share one credential, so the downstream record shows a single caller on every line. When one deletes a production table the culprit is not recoverable from the record, and the only containment available stops all three.

## Your turn

Count the distinct credentials across your agents and divide by the number of agents. Any answer below one is this risk, and the number tells you how many innocent agents a revocation takes down.

---

**Next → [A1.8 · Malicious code execution](https://spbreed.github.io/cyber-commons/lessons/A1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*